# Alternate Plant Analysis Methods

Reference collection of alternate approaches that were tried for isolating plants
and assigning blobs to pots. **None of these are used by the active pipeline.**

The production notebooks (`Process_TrayN.ipynb`) use the simple, reliable method:
mask on the LAB **a** channel, then place one region of interest per pot manually
with `pcv.roi.multi`. That approach is easy to read and predictable.

The methods below were experiments to (a) mask in a different color space, or
(b) place ROIs automatically and automatically drop stray/noise blobs when more
than one blob lands in a single pot. They are kept here for reference so the
processing notebooks stay uncluttered. Each assumes `crop_img` and a cleaned
binary `filled_mask` already exist (same variables as in the processing notebooks).


## 1. HSV threshold mask (alternate to LAB 'a' channel)

Instead of converting to the LAB 'a' channel and thresholding, mask directly in
HSV space with an explicit hue/saturation/value window. Sometimes separates green
tissue from soil better under different lighting, but needs its three ranges tuned.


In [ ]:
# Alternative mask: threshold in HSV space instead of the LAB 'a' channel
thresh_mask, _ = pcv.threshold.custom_range(
    img=crop_img,
    lower_thresh=[10, 50, 70],    # [hue_min, sat_min, val_min]
    upper_thresh=[40, 255, 255],  # [hue_max, sat_max, val_max]
    channel='HSV'
)


## 2. Connected components + auto-grid, nearest-ROI blob filtering

Places ROIs automatically with `pcv.roi.auto_grid` (6 rows x 4 columns), then
labels every separate blob and assigns each to its nearest ROI center. When a pot
gets more than one blob (e.g. a speck of noise next to the plant), it keeps only
the blob closest to the ROI center. Removes the need to hand-enter pot coordinates,
but depends on the auto grid lining up with the real pot layout.


In [ ]:
import cv2
import numpy as np

# Label every separate blob in the mask
num_components, component_labels, stats, centroids_all = cv2.connectedComponentsWithStats(filled_mask)

# ROI centers from an automatically generated grid (6 rows x 4 columns)
rois = pcv.roi.auto_grid(img=filled_mask, mask=filled_mask, nrows=6, ncols=4)
roi_centers = []
for contour_tuple in rois.contours:
    M = cv2.moments(contour_tuple[0])
    if M["m00"] != 0:
        roi_centers.append((int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])))

# Assign every blob to its nearest ROI center
blob_to_roi = {}
for comp_id in range(1, num_components):
    cx, cy = centroids_all[comp_id]
    distances = [np.hypot(cx - rx, cy - ry) for rx, ry in roi_centers]
    nearest = int(np.argmin(distances))
    blob_to_roi.setdefault(nearest, []).append((comp_id, distances[nearest]))

# For pots with more than one blob, keep only the closest to the ROI center
filtered_mask = filled_mask.copy()
for roi_idx, blobs in blob_to_roi.items():
    if len(blobs) > 1:
        for comp_id, dist in sorted(blobs, key=lambda b: b[1])[1:]:
            filtered_mask[component_labels == comp_id] = 0

# Rebuild ROIs / labels on the filtered mask
rois = pcv.roi.auto_grid(img=filtered_mask, mask=filtered_mask, nrows=6, ncols=4)
labeled_mask, num_plants = pcv.create_labels(mask=filtered_mask, rois=rois, roi_type='partial')


## 3. KMeans grid inference, nearest-grid blob filtering

Infers the pot grid from the blobs themselves: clusters blob x-positions into 4
columns with KMeans, then clusters y-positions into 6 rows within each column, to
build 24 grid centers. Every blob is assigned to its nearest grid center, and pots
with multiple blobs keep only the closest. Most automatic of the three, but also
the most fragile: needs enough well-separated blobs for the clustering to find the
true grid, so it can misgroup early in growth or when plants touch.


In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans

# All blob centroids
num_components, component_labels, stats, centroids_all = cv2.connectedComponentsWithStats(filled_mask)
blob_centroids = centroids_all[1:]

# Cluster x into 4 columns
col_kmeans = KMeans(n_clusters=4, random_state=0, n_init='auto').fit(blob_centroids[:, 0].reshape(-1, 1))
col_labels = col_kmeans.labels_
col_centers = sorted(col_kmeans.cluster_centers_.flatten())

# Within each column, cluster y into 6 rows independently -> 24 grid centers
grid_centers = []
for col_idx in range(4):
    col_mask_idx = col_labels == np.argsort(col_kmeans.cluster_centers_.flatten())[col_idx]
    col_blob_y = blob_centroids[col_mask_idx, 1]
    if len(col_blob_y) < 6:
        col_row_centers = np.linspace(col_blob_y.min(), col_blob_y.max(), 6)
    else:
        row_kmeans = KMeans(n_clusters=6, random_state=0, n_init='auto').fit(col_blob_y.reshape(-1, 1))
        col_row_centers = sorted(row_kmeans.cluster_centers_.flatten())
    for y in col_row_centers:
        grid_centers.append((col_centers[col_idx], y))

# Assign every blob to its nearest grid center; keep only the closest per cell
blob_to_grid = {}
for comp_id in range(1, num_components):
    cx, cy = centroids_all[comp_id]
    distances = [np.hypot(cx - rx, cy - ry) for rx, ry in grid_centers]
    nearest = int(np.argmin(distances))
    blob_to_grid.setdefault(nearest, []).append((comp_id, distances[nearest]))

filtered_mask = filled_mask.copy()
for grid_idx, blobs in blob_to_grid.items():
    if len(blobs) > 1:
        for comp_id, dist in sorted(blobs, key=lambda b: b[1])[1:]:
            filtered_mask[component_labels == comp_id] = 0

rois = pcv.roi.auto_grid(img=filtered_mask, mask=filtered_mask, nrows=6, ncols=4)
labeled_mask, num_plants = pcv.create_labels(mask=filtered_mask, rois=rois, roi_type='partial')
